# 🍌 Notebook Klasifikasi Kematangan Pisang
Komparasi Arsitektur **ConvNeXt-Tiny** vs **MobileNetV3-Large** menggunakan *On-The-Fly Augmentation*.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import os

print("TensorFlow Version:", tf.__version__)

In [ ]:
# PENTING: Sesuaikan DATASET_PATH dengan lokasi dataset di Kaggle!
DATASET_PATH = "/kaggle/input/dataset-klasifikasi-pisang-lengkap"

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

print("🚀 Memuat Data Latih (Training)...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

print("\n🚀 Memuat Data Validasi...")
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("\n✅ Kelas yang terdeteksi:", class_names)

In [ ]:
# Augmentasi On-The-Fly (Bekerja otomatis saat training, mati saat evaluasi)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1)
], name="Augmentasi_On_The_Fly")

# Optimasi pipeline dataset agar GPU tidak menunggu disk secara sinkron
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

print("✅ Pipa Data & Augmentasi siap!")

In [ ]:
# 1. Fungsi perakit ConvNeXt-Tiny
def build_convnext_tiny(num_classes):
    base_model = tf.keras.applications.ConvNeXtTiny(
        include_top=False, weights='imagenet', input_shape=(224, 224, 3)
    )
    base_model.trainable = False # Kunci bobot awal
    
    inputs = keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = tf.keras.applications.convnext.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    return keras.Model(inputs, outputs, name="ConvNeXt_Tiny")

# 2. Fungsi perakit MobileNetV3-Large
def build_mobilenet_v3(num_classes):
    base_model = tf.keras.applications.MobileNetV3Large(
        include_top=False, weights='imagenet', input_shape=(224, 224, 3)
    )
    base_model.trainable = False 
    
    inputs = keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = tf.keras.applications.mobilenet_v3.preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    return keras.Model(inputs, outputs, name="MobileNetV3_Large")

In [ ]:
model_convnext = build_convnext_tiny(NUM_CLASSES)
model_mobilenet = build_mobilenet_v3(NUM_CLASSES)

learning_rate = 0.001

model_convnext.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_mobilenet.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model siap dilatih!")
model_convnext.summary()

In [ ]:
EPOCHS = 20

# Callbacks untuk ConvNeXt
callbacks_conv = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_convnext_pisang.h5', monitor='val_accuracy', save_best_only=True)
]

print("🚀 Memulai Training ConvNeXt-Tiny...")
history_convnext = model_convnext.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_conv
)

In [ ]:
# Callbacks untuk MobileNet
callbacks_mob = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_mobilenet_pisang.h5', monitor='val_accuracy', save_best_only=True)
]

print("\n🚀 Memulai Training MobileNetV3-Large...")
history_mobilenet = model_mobilenet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_mob
)

In [ ]:
def plot_history(hist1, hist2, title1, title2):
    plt.figure(figsize=(16, 6))
    
    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(hist1.history['accuracy'], label=f'Train {title1}', color='blue', linestyle='-')
    plt.plot(hist1.history['val_accuracy'], label=f'Val {title1}', color='blue', linestyle='--')
    plt.plot(hist2.history['accuracy'], label=f'Train {title2}', color='red', linestyle='-')
    plt.plot(hist2.history['val_accuracy'], label=f'Val {title2}', color='red', linestyle='--')
    plt.title('Komparasi Akurasi (Accuracy)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    # Plot Loss
    plt.subplot(1, 2, 2)
    plt.plot(hist1.history['loss'], label=f'Train {title1}', color='blue', linestyle='-')
    plt.plot(hist1.history['val_loss'], label=f'Val {title1}', color='blue', linestyle='--')
    plt.plot(hist2.history['loss'], label=f'Train {title2}', color='red', linestyle='-')
    plt.plot(hist2.history['val_loss'], label=f'Val {title2}', color='red', linestyle='--')
    plt.title('Komparasi Kerugian (Loss)')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

plot_history(history_convnext, history_mobilenet, "ConvNeXt", "MobileNetV3")

In [ ]:
print("Mengumpulkan hasil prediksi untuk Confusion Matrix...")
y_true = np.concatenate([np.argmax(labels.numpy(), axis=-1) for images, labels in val_ds])

print("Prediksi menggunakan ConvNeXt-Tiny...")
y_pred_conv = np.argmax(model_convnext.predict(val_ds), axis=-1)

print("Prediksi menggunakan MobileNetV3-Large...")
y_pred_mob = np.argmax(model_mobilenet.predict(val_ds), axis=-1)

def plot_cm(y_t, y_p, title):
    # Memaksa sistem mengakui 3 kelas walau salah satu kosong di validasi
    cm = confusion_matrix(y_t, y_p, labels=range(NUM_CLASSES))
    
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('Label Asli')
    plt.xlabel('Prediksi')
    plt.show()
    
    print(f"\nLaporan Klasifikasi {title}:\n")
    print(classification_report(y_t, y_p, target_names=class_names, labels=range(NUM_CLASSES), zero_division=0))

plot_cm(y_true, y_pred_conv, "Confusion Matrix - ConvNeXt-Tiny")
plot_cm(y_true, y_pred_mob, "Confusion Matrix - MobileNetV3-Large")